# FeynmanEngine — NLO Quickstart

Five-step tour of the NLO machinery, from textbook QED to LHC Drell-Yan.

**Prerequisites**

```bash
pip install feynman-engine
feynman setup     # builds QGRAF + FORM + LoopTools + LHAPDF + OpenLoops 2
```

If you skipped OpenLoops or LHAPDF, the relevant cells will tell you and continue gracefully.

## 1. Textbook QED — `e⁺e⁻ → μ⁺μ⁻` Born σ

The textbook formula `σ = 4πα²/(3s)` — the engine should match it to <1%.

In [ ]:
import math
from feynman_engine.amplitudes.cross_section import total_cross_section

sqrt_s = 91.0
r = total_cross_section("e+ e- -> mu+ mu-", "QED", sqrt_s=sqrt_s)
sigma_textbook = 4 * math.pi * (1/137.036)**2 / (3 * sqrt_s**2) * 0.3894e9

print(f"Engine σ:    {r['sigma_pb']:.4f} pb")
print(f"Textbook σ:  {sigma_textbook:.4f} pb  (Peskin & Schroeder eq. 5.13)")
print(f"Agreement:   {abs(r['sigma_pb'] - sigma_textbook) / sigma_textbook * 100:.2f}%")

## 2. Universal QED NLO — `K = 1 + 3α/(4π)` for any QED process

The engine ships a closed-form universal QED NLO formula based on the charge-correlator structure (Σ Q² × 3/4). For pure-leptonic 2→2 it reproduces the textbook K = 1.001742 exactly; for any other QED process it gives an approximate K with explicit accuracy disclosure.

In [ ]:
from feynman_engine.amplitudes.nlo_qed_general import qed_nlo_kfactor

ALPHA = 1.0 / 137.035999084
K_textbook = 1.0 + 3.0 * ALPHA / (4.0 * math.pi)

for proc in ["e+ e- -> mu+ mu-", "e+ e- -> tau+ tau-", "e+ e- -> e+ e-", "mu+ mu- -> e+ e-"]:
    K = qed_nlo_kfactor(proc, "QED").k_factor
    print(f"  K_QED({proc:25s}) = {K:.6f}  vs textbook {K_textbook:.6f}")

## 3. EW Sudakov LL+NLL — high-energy suppression

At LHC energies √s ≫ M_W, the dominant electroweak NLO correction is the Sudakov double log:
$$\delta_{EW} = -\frac{\alpha}{4\pi \sin^2\theta_W} \sum_{\text{legs}} T_{eff}^2 \times \big(L^2 + 3L\big), \quad L = \log(s/M_W^2)$$

K decreases monotonically with energy — at 1 TeV the suppression is already ~22%.

In [ ]:
from feynman_engine.amplitudes.nlo_ew_general import ew_nlo_sudakov_kfactor

for sqrts in [200, 500, 1000, 2000, 5000]:
    r = ew_nlo_sudakov_kfactor("e+ e- -> mu+ mu-", float(sqrts))
    print(f"  K_EW(e+e-→μμ at √s = {sqrts:>4d} GeV) = {r.k_factor:.4f}")

## 4. Hadronic NLO — pp → DY at 13 TeV from first principles

Catani-Seymour subtraction with HMNvN coefficient functions in MS-bar. Reproduces the Anastasiou et al. K_NLO = 1.21 to within 2%.

In [ ]:
from feynman_engine.amplitudes.nlo_general import hadronic_nlo_drell_yan_v25

try:
    r = hadronic_nlo_drell_yan_v25(sqrt_s_gev=13000.0, m_ll_min=60.0, m_ll_max=120.0)
    print(f"  σ_LO  (γ+Z analytic):  {r.sigma_lo_pb:.2f} pb")
    print(f"  σ_NLO (Vogt + AEM):    {r.sigma_nlo_total_pb:.2f} pb")
    print(f"  K_factor:              {r.k_factor:.4f}  (YR4 reference: K = 1.21)")
    print(f"  Agreement:             {abs(r.k_factor / 1.21 - 1.0) * 100:.1f}%")
except Exception as exc:
    print(f"  Skipping (LHAPDF or PDF set missing): {exc}")

## 5. Trust system in action — refused process

The engine refuses to return a number when no method gives a defensible answer. Instead it returns HTTP 422 with a structured `block_reason` and `workaround`.

In [ ]:
from fastapi.testclient import TestClient
from feynman_engine.api.app import app

client = TestClient(app)
r = client.get("/api/amplitude/cross-section", params={
    "process": "u u~ -> c c~",
    "theory": "QCD",
    "sqrt_s": 91.0,
    "order": "NLO",
})
print(f"HTTP {r.status_code}")
if r.status_code == 422:
    d = r.json()["detail"]
    print(f"trust_level: {d['trust_level']}")
    print(f"block_reason: {d['block_reason'][:100]}…")
    print(f"workaround: {d['workaround'][:120]}…")

## What's next

- **`getting_started.ipynb`** — install + first diagram + spin up the browser UI.
- **`for_undergrad_qft.ipynb`** — teach particle physics with diagram + amplitude examples.
- **`for_lhc_experimentalist.ipynb`** — LHC observables, K-factors, comparison to data.
- **`for_bsm_theorist.ipynb`** — register your own BSM |M̄|² and use it everywhere.
- **Browser UI**: `feynman serve` and visit http://localhost:8000.
- **Swagger docs**: http://localhost:8000/docs.